# NuFrost Local Launcher

Run NuFrost reconstruction locally for a single manually specified image.

In [9]:
from pathlib import Path
import os

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data" / "input"
CACHE_DIR = PROJECT_DIR / "data" / "local_cache"
OUTPUT_DIR = PROJECT_DIR / "data" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"[Info] Project directory: {PROJECT_DIR}")
print(f"[Info] Data directory: {DATA_DIR}")
print(f"[Info] Cache directory: {CACHE_DIR}")
print(f"[Info] Output directory: {OUTPUT_DIR}")

print(f"[Info] Changing working directory to: {PROJECT_DIR}")
os.chdir(str(PROJECT_DIR))

[Info] Project directory: /Users/mckay/Library/CloudStorage/GoogleDrive-yangluhao990714@gmail.com/我的云端硬盘/WorkSpaces/nufrost
[Info] Data directory: /Users/mckay/Library/CloudStorage/GoogleDrive-yangluhao990714@gmail.com/我的云端硬盘/WorkSpaces/nufrost/data/input
[Info] Cache directory: /Users/mckay/Library/CloudStorage/GoogleDrive-yangluhao990714@gmail.com/我的云端硬盘/WorkSpaces/nufrost/data/cache
[Info] Output directory: /Users/mckay/Library/CloudStorage/GoogleDrive-yangluhao990714@gmail.com/我的云端硬盘/WorkSpaces/nufrost/data/output
[Info] Changing working directory to: /Users/mckay/Library/CloudStorage/GoogleDrive-yangluhao990714@gmail.com/我的云端硬盘/WorkSpaces/nufrost


In [10]:
import src.data_loader
import importlib
import src

importlib.reload(src)

importlib.reload(src.data_loader)


<module 'src.data_loader' from '/Users/mckay/Library/CloudStorage/GoogleDrive-yangluhao990714@gmail.com/我的云端硬盘/WorkSpaces/nufrost/src/data_loader.py'>

In [11]:
# Modify these parameters manually
TARGET_TIME = "2023-06-15T00:00:00"
TARGET_LON = 91.2734
TARGET_LAT = 29.7904
TARGET_BAND = "BLUE"
HLS_DATA_DIR = PROJECT_DIR / "data/hls"  # Adjust path as needed

IMAGE_NAMES = []

if not IMAGE_NAMES:
    from src.data_loader import find_image_chunks
    image_paths = find_image_chunks(str(HLS_DATA_DIR), TARGET_LON, TARGET_LAT, TARGET_BAND)
    image_paths_list = [image_paths] if image_paths else []
    print(f"[Info] Auto-detected {len(image_paths)} VRT chunk(s).")
else:
    image_paths_list = [[str(DATA_DIR / name)] for name in IMAGE_NAMES]
    print(f"[Info] Number of images to process: {len(IMAGE_NAMES)}")


[Info] Auto-detected 1 image chunks.


In [12]:
print("========== Starting Local NuFrost Reconstruction ==========")

recons = []
for image_paths in image_paths_list:
    # We use the first file to derive the output path
    first_path = Path(image_paths[0])
    output_path = OUTPUT_DIR / f"{first_path.stem}_{TARGET_TIME.replace(':', '-')}__nufrost.tif"

    print(f"\n--- Processing: {first_path.name} ---")

    recon = src.reconstruct_nufrost(
        image=image_paths,
        target_time=TARGET_TIME,
        output_path=str(output_path),
        cache_dir=str(CACHE_DIR),
        n_jobs=-1,
        force_refresh=False,
    )

    recons.append((first_path.name, recon))
    print(f"[Success] Reconstruction shape: {recon.shape}")
    print(f"[Success] Saved to: {output_path}")


========== Starting Local NuFrost Reconstruction ==========

--- Processing: BLUE_lon91.2734_lat29.7904_part1.vrt ---
[System] Reading 4 TIFF(s) into memory...


RasterioIOError: Read failed. See previous exception for details.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

for image_name, recon in recons:
    plt.figure(figsize=(8, 8))
    plt.imshow(recon, cmap="viridis")
    plt.colorbar(shrink=0.8)
    plt.title(f"NuFrost Reconstruction: {image_name}")
    plt.axis("off")
    plt.show()
